# Day 55: Async and await
Learn to overlap waiting, understand threads, and see how a coroutine pauses and resumes.

**Python 3.9+**, Jupyter Python kernel. Run top to bottom. Standard library only; no network or setup files. Examples take a few seconds.

## 1. Why async?
Three report requests may spend most of their time waiting. Start them together and let their waits overlap.

- `async def` defines a coroutine function; calling it creates a coroutine object.
- `await` obtains an awaitable's result, pausing the current coroutine if needed.
- An **event loop** runs other ready tasks while one waits.

This helps with waiting for I/O. It does not automatically speed up calculations.

In [ ]:
import asyncio
import dis
import inspect
import threading
import time
from concurrent.futures import ThreadPoolExecutor

DELAY = 0.3  # Simulated I/O; no real API calls.

## 2. Calling is different from running
A regular function runs when called. Calling an `async def` function creates a coroutine; its body starts when awaited or scheduled.

In [ ]:
async def greet(name):
    print("Body started")
    await asyncio.sleep(0.1)
    return f"Hello, {name}!"

pending = greet("Aadhi")
print("Before await:", inspect.getcoroutinestate(pending))
print(await pending)
print("After await:", inspect.getcoroutinestate(pending))

Expect `CORO_CREATED`, then the body output, then `CORO_CLOSED`. A completed coroutine cannot be awaited again: call `greet(...)` to create another.

## 3. Sequential versus concurrent
Awaiting each call in a loop is sequential. `gather()` schedules independent coroutines together; its results stay in input order.

In [ ]:
async def fetch_report(name):
    await asyncio.sleep(DELAY)
    return f"{name}: ready"

start = time.perf_counter()
sequential = []
for name in ["sales", "stock", "returns"]:
    sequential.append(await fetch_report(name))
print("Sequential:", sequential)
print(f"Elapsed: {time.perf_counter() - start:.2f}s (about {3 * DELAY:.1f}s)")

start = time.perf_counter()
concurrent = await asyncio.gather(
    fetch_report("sales"), fetch_report("stock"), fetch_report("returns"),
)
print("Concurrent:", concurrent)
print(f"Elapsed: {time.perf_counter() - start:.2f}s (about {DELAY:.1f}s)")
assert sequential == concurrent

```text
Sequential: sales [wait] -> stock [wait] -> returns [wait]
Concurrent: sales   [wait] -> done
            stock   [wait] -> done
            returns [wait] -> done
```
Times are approximate: overlapping delays, not faster CPU execution.

## 4. Tasks and one event-loop thread
A **coroutine** describes resumable work. A **Task** schedules and tracks it. A **Future** represents a result that will become available later.

Both tasks below run on the same thread. They take turns when they yield. Exact output order can vary.

In [ ]:
async def steps(label):
    for step in range(1, 3):
        print(label, "step", step, "thread:", threading.current_thread().name)
        await asyncio.sleep(0.05)
    return f"{label} finished"

task_a = asyncio.create_task(steps("A"))
task_b = asyncio.create_task(steps("B"))
print(await task_a)
print(await task_b)  # B was already scheduled.
assert task_a.done() and task_b.done()

## 5. Mistake: blocking inside async
`time.sleep()` blocks the thread. `asyncio.sleep()` pauses the task and lets the loop continue. Adding `async` does not convert blocking calls.

Watch whether the heartbeat runs **during** each wait.

In [ ]:
async def heartbeat():
    for tick in range(3):
        print("  heartbeat", tick)
        await asyncio.sleep(0.05)

async def blocking_wait():
    print("Blocking wait starts")
    time.sleep(0.2)  # Deliberately blocks the loop.
    print("Blocking wait ends")

async def friendly_wait():
    print("Async wait starts")
    await asyncio.sleep(0.2)
    print("Async wait ends")

print("BLOCKING:")
await asyncio.gather(blocking_wait(), heartbeat())
print("\nCOOPERATIVE:")
await asyncio.gather(friendly_wait(), heartbeat())

## 6. Threading models

| Model | Who switches work? | Good fit |
|---|---|---|
| Async tasks, one loop thread | Event loop when tasks yield | Many waits with async libraries |
| Multiple OS threads | Operating system | Blocking I/O libraries |
| Multiple processes | OS; separate Python runtimes | Heavy Python calculations |

Ordinary CPython builds have a **GIL**: only one thread executes Python bytecode at a time. I/O and some native libraries release it. Optional free-threaded builds differ. Threads share memory; shared mutable data may need locks.

The same blocking function can run in a thread pool or through `asyncio.to_thread()`.

In [ ]:
def read_legacy_report(name):
    time.sleep(0.15)  # Pretend this is a blocking I/O library.
    return name, threading.current_thread().name

# Synchronous code: collecting results blocks the caller.
with ThreadPoolExecutor(max_workers=2) as pool:
    results = list(pool.map(read_legacy_report, ["sales", "stock"]))
print("Thread pool:", results)

# Async code: move blocking I/O to a worker so the loop stays available.
print("Event-loop thread:", threading.current_thread().name)
results = await asyncio.gather(
    asyncio.to_thread(read_legacy_report, "returns"),
    heartbeat(),
)
print("Worker result:", results[0])

## 7. Can await run directly in Python main code?

| Place | Entry point |
|---|---|
| Normal `.py` file, module level | `asyncio.run(main())` |
| Inside `async def` | `await operation()` |
| Inside ordinary `def` | Direct `await` is a syntax error |
| Jupyter/IPython with autoawait | Top-level `await` works |

**Why an async function?** Waiting needs a resumable context that retains local variables and the instruction position. Normal Python script syntax restricts `await` to async functions. Jupyter supplies special compilation and execution support for cells.

`main` is a conventional name, not a keyword. `asyncio.run()` manages a new event loop and closes it afterward. Do not call it on a thread already running a loop, as in a typical notebook.

In [ ]:
# Demonstrate the script syntax rule without breaking the notebook.
try:
    compile("await asyncio.sleep(0.1)", "example.py", "exec")
except SyntaxError as exc:
    print(type(exc).__name__ + ":", exc.msg)

async def main():
    results = await asyncio.gather(fetch_report("sales"), fetch_report("stock"))
    print(results)

await main()  # Notebook entry point.

### Complete standalone program
Copy into `async_demo.py` and run `python async_demo.py`. This is a script example, not a cell to run inside this notebook.

```python
import asyncio

async def fetch(name):
    await asyncio.sleep(0.3)
    return f"{name}: ready"

async def main():
    print(await asyncio.gather(fetch("sales"), fetch("stock")))

if __name__ == "__main__":
    asyncio.run(main())
```
The guard starts the program when executed directly, without starting it when another file imports it.

## 8. What happens underneath?
Python compiles `async def` into bytecode marked as a coroutine. It does not generate another source file or create a thread.

1. A call creates a coroutine object with resumable execution state.
2. A Task drives the coroutine on the event loop.
3. An unfinished awaitable can suspend it, retaining locals and its instruction position.
4. Completion makes the task ready to resume with a result or exception.

`await` does **not always suspend**: an available result can continue immediately. `asyncio.sleep()` yields; positive delays use a timer. Real async I/O uses OS readiness/completion notifications.

### Observe saved state
An Event holds the worker at `await` until released. This reveals its retained local variable.

In [ ]:
async def saved_state(gate, started):
    value = 10
    started.set()
    await gate.wait()
    return value + 5

gate = asyncio.Event()
started = asyncio.Event()
coro = saved_state(gate, started)
print("Before scheduling:", inspect.getcoroutinestate(coro))
task = asyncio.create_task(coro)
await started.wait()
try:
    print("While waiting:", inspect.getcoroutinestate(coro))
    print("Saved local value:", coro.cr_frame.f_locals["value"])
finally:
    gate.set()
    result = await task
print("Result:", result)
print("After completion:", inspect.getcoroutinestate(coro))
assert result == 15

### Actual compiler output (optional)
`dis` displays bytecode, not generated Python source. Depending on your CPython version, look for `GET_AWAITABLE`, `SEND`, `YIELD_VALUE`, or `RESUME`. Names and layout vary by version.

Conceptually: **run -> save position -> suspend -> resume -> return**. No need to memorize instructions.

In [ ]:
async def tiny():
    await asyncio.sleep(0)
    return 7

dis.dis(tiny)

## 9. Practice
Set `DELAY = 0.5` and rerun section 3. Predict **1.5s sequential** and **0.5s concurrent**.

Use direct `await` when the next step needs the result. Schedule independent operations together to overlap waits. Use async libraries or `to_thread()` for blocking I/O; keep heavy CPU work off the loop.

**References:** [Coroutines and tasks](https://docs.python.org/3/library/asyncio-task.html), [asyncio runners](https://docs.python.org/3/library/asyncio-runner.html), [Threading and the GIL](https://docs.python.org/3/library/threading.html), [IPython autoawait](https://ipython.readthedocs.io/en/stable/interactive/autoawait.html), [Bytecode with dis](https://docs.python.org/3/library/dis.html).